In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Set torch cache directory

In [3]:
# Import the necessary packages
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn.functional as F
import random
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torchvision import models
import networkx as nx
from pyvis.network import Network
from collections import defaultdict

In [4]:
# -----------------------------
# MLP Wrapper
# -----------------------------
class MLPWrapper(nn.Module):
    def __init__(self, layer, hidden_dim=128):
        super().__init__()
        self.layer = layer
        self.out_channels = layer.out_channels
        self.mlp = nn.Sequential(
            nn.Linear(self.out_channels, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, self.out_channels)
        )

    def forward(self, x):
        x = self.layer(x)
        B, C, H, W = x.shape
        x_perm = x.permute(0, 2, 3, 1).contiguous()
        x_flat = x_perm.view(-1, C)
        x_mlp = self.mlp(x_flat)
        return x_mlp.view(B, H, W, C).permute(0, 3, 1, 2)

In [5]:
# -----------------------------
# Wrap VGG
# -----------------------------
def wrap_vgg_with_mlp(vgg_model):
    new_layers = []
    for layer in vgg_model.features:
        if isinstance(layer, nn.Conv2d):
            new_layers.append(MLPWrapper(layer))
        else:
            new_layers.append(layer)
    vgg_model.features = nn.Sequential(*new_layers)
    return vgg_model

In [7]:
# -----------------------------
# Model
# -----------------------------
class VGG_MLP(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        base = models.vgg16(weights=None)
        base = wrap_vgg_with_mlp(base)
        self.features = base.features
        self.avgpool = base.avgpool
        self.classifier = nn.Sequential(
            nn.Linear(512*7*7, 4096), nn.ReLU(),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [8]:
# -----------------------------
# Activation Cache
# -----------------------------
class ActivationCache:
    def __init__(self):
        self.cache = {}

    def hook(self, name):
        def fn(module, inp, out):
            self.cache[name] = out.detach()
        return fn

In [9]:

# -----------------------------
# Full Causal Circuit Tracer
# -----------------------------
class CausalCircuitTracer:
    def __init__(self, model):
        self.model = model

    def run_with_cache(self, x):
        cache = ActivationCache()
        handles = []
        for name, module in self.model.named_modules():
            if isinstance(module, MLPWrapper):
                handles.append(module.register_forward_hook(cache.hook(name)))

        out = self.model(x)

        for h in handles:
            h.remove()

        return out, cache.cache

    # -----------------------------
    # Patch ONE neuron
    # -----------------------------
    def patch_neuron(self, x, layer_name, neuron_idx):
        def hook(module, inp, out):
            out = out.clone()
            out[:, neuron_idx] = 0
            return out

        handles = []
        for name, module in self.model.named_modules():
            if name == layer_name:
                handles.append(module.register_forward_hook(hook))

        out = self.model(x)

        for h in handles:
            h.remove()

        return out
    
    # -----------------------------
    # Patch neuron and observe another neuron
    # -----------------------------
    def patch_and_measure(self, x, src_layer, src_idx, tgt_layer, tgt_idx):
        cache = {}

        def src_hook(module, inp, out):
            out = out.clone()
            out[:, src_idx] = 0
            return out

        def tgt_hook(module, inp, out):
            cache['tgt'] = out.detach()

        handles = []

        for name, module in self.model.named_modules():
            if name == src_layer:
                handles.append(module.register_forward_hook(src_hook))
            if name == tgt_layer:
                handles.append(module.register_forward_hook(tgt_hook))

        _ = self.model(x)

        for h in handles:
            h.remove()

        return cache['tgt']

    # -----------------------------
    # Compute FULL causal graph
    # -----------------------------
    def compute_causal_graph(self, x, target_class, node_threshold=0.01, edge_threshold=0.005):
        base_out, cache = self.run_with_cache(x)
        base_score = base_out[0, target_class]

        graph = nx.DiGraph()
        layers = list(cache.keys())

        # ---- Node importance ----
        node_importance = {}
        for layer in layers:
            act = cache[layer]
            C = act.shape[1]

            for c in range(C):
                patched_out = self.patch_neuron(x, layer, c)
                effect = (base_score - patched_out[0, target_class]).item()

                if abs(effect) > node_threshold:
                    node = f"{layer}_ch{c}"
                    node_importance[node] = effect
                    graph.add_node(node, layer=layer, importance=effect)

        # ---- Edge causality (TRUE) ----
        for i in range(len(layers) - 1):
            l1, l2 = layers[i], layers[i+1]

            for n1 in range(cache[l1].shape[1]):
                node1 = f"{l1}_ch{n1}"
                if node1 not in node_importance:
                    continue

                for n2 in range(cache[l2].shape[1]):
                    node2 = f"{l2}_ch{n2}"
                    if node2 not in node_importance:
                        continue

                    base_tgt = cache[l2][0, n2].mean()
                    patched_tgt = self.patch_and_measure(x, l1, n1, l2, n2)[0, n2].mean()

                    effect = (base_tgt - patched_tgt).item()

                    if abs(effect) > edge_threshold:
                        graph.add_edge(node1, node2, weight=effect)

        return graph

    # -----------------------------
    # Class-wise circuits
    # -----------------------------
    def compute_classwise_circuits(self, x, num_classes):
        circuits = {}
        for cls in range(num_classes):
            graph = self.compute_causal_graph(x, cls)
            circuits[cls] = graph
        return circuits

In [10]:
# -----------------------------
# Interactive Visualization
# -----------------------------
def visualize_graph(graph, filename="circuit.svg"):
    net = Network(directed=True)

    for node, data in graph.nodes(data=True):
        net.add_node(node, title=str(data), value=abs(data.get("importance", 1)))

    for u, v, data in graph.edges(data=True):
        net.add_edge(u, v, value=abs(data.get("weight", 1)))

    net.show(filename)